# Exp7.2.5 — Output Synaptic Alpha

Analysis-only notebook for the frozen-L2 readout and paired end-to-end `alpha_out` ablations.

Primary comparison: `alpha_out=0` vs `alpha_out=0.5` with `beta_out=0.5`. Training and checkpoint selection use valid length. Full-256 metrics are diagnostics only.

The notebook intentionally presents `task_only` first and `task_plus_reg` second.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repository root not found")


REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "notebooks" / "artifacts" / "experiment_7_2_5_output_synaptic_alpha" / "output_synaptic_alpha_v1"
print(ROOT)


## 1. Task-only — frozen L2 output-head refit

Rows are the historical S1-S4 L2 representations. The alpha gains isolate output readout dynamics because L1/L2 are frozen.


In [ ]:
task_only_frozen = pd.read_csv(ROOT / "report_frozen_task_only.csv")
display(task_only_frozen)


## 2. Task-only — paired end-to-end WholeCount CE


In [ ]:
task_only_e2e_wc = pd.read_csv(ROOT / "report_e2e_task_only_wc.csv")
display(task_only_e2e_wc)


## 3. Task-only — paired end-to-end timestep CE


In [ ]:
task_only_e2e_tsce = pd.read_csv(ROOT / "report_e2e_task_only_tsce.csv")
display(task_only_e2e_tsce)


## 4. Task + regularizer — frozen L2 output-head refit


In [ ]:
reg_frozen = pd.read_csv(ROOT / "report_frozen_task_plus_reg.csv")
display(reg_frozen)


## 5. Task + regularizer — paired end-to-end WholeCount CE


In [ ]:
reg_e2e_wc = pd.read_csv(ROOT / "report_e2e_task_plus_reg_wc.csv")
display(reg_e2e_wc)


## 6. Task + regularizer — paired end-to-end timestep CE


In [ ]:
reg_e2e_tsce = pd.read_csv(ROOT / "report_e2e_task_plus_reg_tsce.csv")
display(reg_e2e_tsce)


## 7. Paired alpha gains

These tables retain architecture, objective and regularization, so alpha effects can be checked without averaging incompatible conditions.


In [ ]:
frozen_delta = pd.read_csv(ROOT / "frozen_alpha_delta_summary.csv")
e2e_delta = pd.read_csv(ROOT / "e2e_alpha_delta_summary.csv")
probe_delta = pd.read_csv(ROOT / "e2e_probe_alpha_delta_summary.csv")

frozen_ba = frozen_delta[frozen_delta.metric == "balanced_accuracy"].copy()
e2e_ba = e2e_delta[e2e_delta.metric == "balanced_accuracy"].copy()
probe_ba = probe_delta[probe_delta.metric == "balanced_accuracy"].copy()

for frame in (frozen_ba, e2e_ba, probe_ba):
    frame["delta_pp_mean"] = 100.0 * frame["delta_mean"]

print("Frozen-L2 alpha gain")
display(frozen_ba)
print("End-to-end native alpha gain")
display(e2e_ba)
print("End-to-end L2-probe alpha gain")
display(probe_ba)


## 8. Architecture-specific end-to-end alpha gain


In [ ]:
arch = e2e_ba.pivot_table(
    index=["regularization", "objective"],
    columns="architecture",
    values="delta_pp_mean",
)
display(arch)

for regularization in ("task_only", "task_plus_reg"):
    subset = e2e_ba[e2e_ba.regularization == regularization]
    plot = subset.pivot(index="objective", columns="architecture", values="delta_pp_mean")
    ax = plot.plot(kind="bar", figsize=(8, 4))
    ax.axhline(0, linewidth=1)
    ax.set_ylabel("alpha=.5 - alpha=0 (BA pp)")
    ax.set_title(f"E2E alpha gain — {regularization}")
    plt.tight_layout()
    plt.show()


## 9. Tail diagnostics

Full-window behavior is diagnostic only; valid-length WholeCount BA remains the primary deployment metric.


In [ ]:
frozen_tail = pd.read_csv(ROOT / "frozen_head_tail_summary.csv")
e2e_tail = pd.read_csv(ROOT / "e2e_tail_summary.csv")

print("Frozen head tail activity — test")
display(frozen_tail[frozen_tail.split == "test"])
print("E2E tail activity — test")
display(e2e_tail[e2e_tail.split == "test"])


## 10. Readout vs representation-shaping diagnostic

For WholeCount, the end-to-end alpha gain is compared with frozen S1; for timestep CE it is compared with frozen S2. `shaping_diagnostic` is descriptive and should not be treated as a strict additive causal decomposition.


In [ ]:
mechanism = pd.read_csv(ROOT / "mechanism_alpha_gain_summary.csv")
for column in ("e2e_alpha_gain_mean", "frozen_alpha_gain_mean", "shaping_diagnostic_mean"):
    mechanism[column.replace("_mean", "_pp")] = 100.0 * mechanism[column]
display(mechanism)
